In [ ]:
__trill_node__ = {
    "id": "7d41f188-ec81-4642-ba73-eeb46ae8ebe2",
    "type": "DATA_LOADING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    import pandas as pd

    # Load the CSV directly
    df = pd.read_csv("data/Energy_Usage_5000.csv")

    # Select relevant columns and clean missing values
    grouped_data = df[["BUILDING TYPE", "TOTAL KWH", "TOTAL THERMS"]].dropna()

    # Return cleaned DataFrame
    return grouped_data

_curio_output = _curio_node()

try:
    data_7d41f188_ec81_4642_ba73_eeb46ae8ebe2 = _curio_output
except NameError:
    data_7d41f188_ec81_4642_ba73_eeb46ae8ebe2 = None


In [ ]:
__trill_node__ = {
    "id": "9eafa9de-726a-404b-8c97-d3d5ec94e51d",
    "type": "DATA_CLEANING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = data_7d41f188_ec81_4642_ba73_eeb46ae8ebe2
    arg = input_0

    def remove_outliers(df, column):
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        return df[(df[column] >= Q1 - 1.5 * IQR) & (df[column] <= Q3 + 1.5 * IQR)]

    def clean(df):
        # Drop only if columns exist
        required_cols = ['CENSUS BLOCK', 'BUILDING TYPE', 'BUILDING_SUBTYPE']
        drop_cols = [col for col in required_cols if col in df.columns]

        df_cleaned = df.dropna(subset=drop_cols).copy()

        # Standard KWH/THERM fill
        kwh_columns = [col for col in df.columns if 'KWH' in col and '2010' in col and 'SQFT' not in col]
        therm_columns = [col for col in df.columns if 'THERM' in col and '2010' in col and 'SQFT' not in col]
        df_cleaned[kwh_columns] = df_cleaned[kwh_columns].fillna(df_cleaned[kwh_columns].median())
        df_cleaned[therm_columns] = df_cleaned[therm_columns].fillna(df_cleaned[therm_columns].median())

        for col in [
            'TOTAL KWH', 'TOTAL THERMS',
            'OCCUPIED UNITS PERCENTAGE', 'OCCUPIED UNITS',
            'RENTER-OCCUPIED HOUSING UNITS'
        ]:
            if col in df_cleaned.columns:
                df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

        df_cleaned['ELECTRICITY ACCOUNTS'] = pd.to_numeric(df_cleaned.get('ELECTRICITY ACCOUNTS'), errors='coerce')
        df_cleaned['GAS ACCOUNTS'] = pd.to_numeric(df_cleaned.get('GAS ACCOUNTS'), errors='coerce')
        df_cleaned['ELECTRICITY'] = df_cleaned['TOTAL KWH']
        df_cleaned['GAS'] = df_cleaned['TOTAL THERMS']

        df_cleaned = df_cleaned.loc[:, df_cleaned.isnull().mean() < 0.2]

        if 'TERM APRIL 2010' in df.columns:
            df.rename(columns={'TERM APRIL 2010': 'THERM APRIL 2010'}, inplace=True)

        # Standardize community names
        if 'COMMUNITY AREA NAME' in df_cleaned.columns:
            df_cleaned['COMMUNITY AREA NAME'] = df_cleaned['COMMUNITY AREA NAME'].str.strip().str.upper()
            df_cleaned['COMMUNITY AREA NAME'] = df_cleaned['COMMUNITY AREA NAME'].replace({
                "LAKEVIEW": "LAKE VIEW",
                "O'HARE": "OHARE"
            })

        # Ensure total columns are present
        if 'TOTAL KWH' in df_cleaned.columns:
            df_cleaned['TOTAL KWH'] = df_cleaned['TOTAL KWH'].fillna(df_cleaned['TOTAL KWH'].median())
        if 'TOTAL THERMS' in df_cleaned.columns:
            df_cleaned['TOTAL THERMS'] = df_cleaned['TOTAL THERMS'].fillna(df_cleaned['TOTAL THERMS'].median())

        if 'AVERAGE BUILDING AGE' in df_cleaned.columns:
            df_cleaned['DECADE BUILT'] = (2010 - df_cleaned['AVERAGE BUILDING AGE']) // 10 * 10

        df_cleaned = remove_outliers(df_cleaned, 'TOTAL KWH')
        df_cleaned = remove_outliers(df_cleaned, 'TOTAL THERMS')



        return df_cleaned


    # Run cleaning and return
    return clean(arg)


_curio_output = _curio_node()

try:
    result_9eafa9de_726a_404b_8c97_d3d5ec94e51d = _curio_output
except NameError:
    result_9eafa9de_726a_404b_8c97_d3d5ec94e51d = None


In [ ]:
__trill_node__ = {
    "id": "fc8f5cba-2ecd-4702-b19f-d471b49104c8",
    "type": "DATA_TRANSFORMATION",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_9eafa9de_726a_404b_8c97_d3d5ec94e51d
    arg = input_0

    # Assume `arg` is the cleaned DataFrame from the previous card
    import pandas as pd

    energy_long = pd.melt(
        arg,
        id_vars='BUILDING TYPE',
        value_vars=['TOTAL KWH', 'TOTAL THERMS'],
        var_name='ENERGY TYPE',
        value_name='VALUE'
    )

    total_by_type = energy_long.groupby('BUILDING TYPE')['VALUE'].transform('sum')
    energy_long['PERCENTAGE'] = (energy_long['VALUE'] / total_by_type) * 100

    return energy_long


_curio_output = _curio_node()

try:
    result_fc8f5cba_2ecd_4702_b19f_d471b49104c8 = _curio_output
except NameError:
    result_fc8f5cba_2ecd_4702_b19f_d471b49104c8 = None


In [ ]:
__trill_node__ = {
    "id": "53e8c833-202d-40d2-8ccf-d0b304566593",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_fc8f5cba_2ecd_4702_b19f_d471b49104c8

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "data": { "name": "energy_transformed_1" },
      "mark": "rect",
      "encoding": {
        "x": { "field": "BUILDING TYPE", "type": "nominal" },
        "y": { "field": "ENERGY TYPE", "type": "nominal" },
        "color": {
          "field": "VALUE",
          "type": "quantitative",
          "scale": { "scheme": "viridis" }
        },
        "tooltip": [
          { "field": "BUILDING TYPE" },
          { "field": "ENERGY TYPE" },
          { "field": "VALUE", "format": ".2f" },
          { "field": "PERCENTAGE", "format": ".1f" }
        ]
      },
      "title": "Energy Consumption Heatmap (KWH + THERMS)"
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_53e8c833_202d_40d2_8ccf_d0b304566593 = _curio_output
except NameError:
    result_53e8c833_202d_40d2_8ccf_d0b304566593 = None


In [ ]:
__trill_node__ = {
    "id": "22d68832-0b7b-4a7c-8f11-989d4780f56f",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_fc8f5cba_2ecd_4702_b19f_d471b49104c8

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "data": { "name": "energy_transformed_1" },
      "mark": "circle",
      "encoding": {
        "x": {
          "field": "BUILDING TYPE",
          "type": "nominal",
          "axis": { "labelAngle": -45 }
        },
        "y": { "field": "VALUE", "type": "quantitative" },
        "color": { "field": "ENERGY TYPE", "type": "nominal" },
        "size": { "field": "VALUE", "type": "quantitative" },
        "tooltip": [
          { "field": "BUILDING TYPE" },
          { "field": "ENERGY TYPE" },
          { "field": "VALUE", "format": ".2f" }
        ]
      },
      "title": "Dot Plot of Energy Usage by Building Type"
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_22d68832_0b7b_4a7c_8f11_989d4780f56f = _curio_output
except NameError:
    result_22d68832_0b7b_4a7c_8f11_989d4780f56f = None


In [ ]:
__trill_node__ = {
    "id": "7a1c0e5b-3c39-4727-a616-f4ca97fdbb44",
    "type": "DATA_TRANSFORMATION",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_9eafa9de_726a_404b_8c97_d3d5ec94e51d
    arg = input_0

    # Group by building type and compute average gas usage
    df_avg_gas = arg.groupby("BUILDING TYPE")["TOTAL THERMS"].mean().reset_index()
    df_avg_gas.rename(columns={"TOTAL THERMS": "AVG TOTAL THERMS"}, inplace=True)

    return df_avg_gas


_curio_output = _curio_node()

try:
    result_7a1c0e5b_3c39_4727_a616_f4ca97fdbb44 = _curio_output
except NameError:
    result_7a1c0e5b_3c39_4727_a616_f4ca97fdbb44 = None


In [ ]:
__trill_node__ = {
    "id": "ce74522c-d689-4cf9-bb41-e4f6a24882d4",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_7a1c0e5b_3c39_4727_a616_f4ca97fdbb44

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "data": { "name": "avg_gas_by_building" },
      "mark": "bar",
      "encoding": {
        "x": {
          "field": "BUILDING TYPE",
          "type": "nominal",
          "axis": { "labelAngle": -45 }
        },
        "y": { "field": "AVG TOTAL THERMS", "type": "quantitative" },
        "tooltip": [
          { "field": "BUILDING TYPE" },
          { "field": "AVG TOTAL THERMS", "format": ".2f" }
        ],
        "color": { "field": "BUILDING TYPE", "type": "nominal" }
      }
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_ce74522c_d689_4cf9_bb41_e4f6a24882d4 = _curio_output
except NameError:
    result_ce74522c_d689_4cf9_bb41_e4f6a24882d4 = None


In [ ]:
__trill_node__ = {
    "id": "15aa8924-3c5a-42a3-a89e-456f675c469a",
    "type": "DATA_LOADING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    import pandas as pd

    df = pd.read_csv("data/Energy_Usage_5000.csv")

    # Standardize column names right away for consistency
    df.columns = [col.upper().strip() for col in df.columns]

    # Just return full dataset for now no filtering yet
    return df

_curio_output = _curio_node()

try:
    data_15aa8924_3c5a_42a3_a89e_456f675c469a = _curio_output
except NameError:
    data_15aa8924_3c5a_42a3_a89e_456f675c469a = None


In [ ]:
__trill_node__ = {
    "id": "23a15771-4d9a-479f-911f-b6f758850574",
    "type": "DATA_CLEANING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = data_15aa8924_3c5a_42a3_a89e_456f675c469a
    arg = input_0

    def remove_outliers(df, column):
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        return df[(df[column] >= Q1 - 1.5 * IQR) & (df[column] <= Q3 + 1.5 * IQR)]

    def clean(df):
        # We assume column names are already uppercased by the data loading card
        required_cols = ['COMMUNITY AREA NAME', 'TOTAL KWH', 'TOTAL THERMS', 'BUILDING TYPE']
        df = df.dropna(subset=required_cols).copy()

        df['COMMUNITY AREA NAME'] = df['COMMUNITY AREA NAME'].str.strip().str.upper()
        df['TOTAL KWH'] = df['TOTAL KWH'].fillna(df['TOTAL KWH'].median())
        df['TOTAL THERMS'] = df['TOTAL THERMS'].fillna(df['TOTAL THERMS'].median())

        df = remove_outliers(df, 'TOTAL KWH')
        df = remove_outliers(df, 'TOTAL THERMS')

        return df

    return clean(arg)


_curio_output = _curio_node()

try:
    result_23a15771_4d9a_479f_911f_b6f758850574 = _curio_output
except NameError:
    result_23a15771_4d9a_479f_911f_b6f758850574 = None


In [ ]:
__trill_node__ = {
    "id": "2cd9e67d-4e07-4852-8a55-f6b50cd3658d",
    "type": "DATA_TRANSFORMATION",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_23a15771_4d9a_479f_911f_b6f758850574
    arg = input_0

    df_avg = arg[["COMMUNITY AREA NAME", "TOTAL KWH", "TOTAL THERMS"]].dropna()

    agg_df = df_avg.groupby("COMMUNITY AREA NAME").agg({
        "TOTAL KWH": "mean",
        "TOTAL THERMS": "mean"
    }).reset_index()

    agg_df["AVG ENERGY USE"] = agg_df["TOTAL KWH"] + agg_df["TOTAL THERMS"]

    top10 = agg_df.sort_values("AVG ENERGY USE", ascending=False).head(10)

    return top10


_curio_output = _curio_node()

try:
    result_2cd9e67d_4e07_4852_8a55_f6b50cd3658d = _curio_output
except NameError:
    result_2cd9e67d_4e07_4852_8a55_f6b50cd3658d = None


In [ ]:
__trill_node__ = {
    "id": "4e7a7f96-4615-4099-bbb9-b5811b560361",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_2cd9e67d_4e07_4852_8a55_f6b50cd3658d

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "data": { "name": "top10_avg_energy_by_community" },
      "mark": "bar",
      "encoding": {
        "x": {
          "field": "COMMUNITY AREA NAME",
          "type": "nominal",
          "sort": "-y",
          "axis": { "labelAngle": -45 }
        },
        "y": {
          "field": "AVG ENERGY USE",
          "type": "quantitative",
          "title": "Avg Energy Use (KWH + THERMS)"
        },
        "tooltip": [
          { "field": "COMMUNITY AREA NAME" },
          { "field": "AVG ENERGY USE", "format": ".2f" }
        ],
        "color": {
          "field": "AVG ENERGY USE",
          "type": "quantitative",
          "scale": { "scheme": "blues" }
        }
      },
      "title": "Top 10 Communities by Avg Energy Consumption"
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_4e7a7f96_4615_4099_bbb9_b5811b560361 = _curio_output
except NameError:
    result_4e7a7f96_4615_4099_bbb9_b5811b560361 = None


In [ ]:
__trill_node__ = {
    "id": "0957871c-b1f3-4c89-84de-3dc8f88e49c7",
    "type": "DATA_TRANSFORMATION",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_23a15771_4d9a_479f_911f_b6f758850574
    arg = input_0

    df_scatter = arg[["TOTAL KWH", "TOTAL THERMS", "BUILDING TYPE"]].dropna()
    df_scatter = df_scatter[(df_scatter["TOTAL KWH"] > 0) & (df_scatter["TOTAL THERMS"] > 0)]

    return df_scatter


_curio_output = _curio_node()

try:
    result_0957871c_b1f3_4c89_84de_3dc8f88e49c7 = _curio_output
except NameError:
    result_0957871c_b1f3_4c89_84de_3dc8f88e49c7 = None


In [ ]:
__trill_node__ = {
    "id": "2ba732cf-7fa3-4a59-b41f-faa3d707994f",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_0957871c_b1f3_4c89_84de_3dc8f88e49c7

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "data": { "name": "scatter_energy_usage" },
      "mark": "point",
      "encoding": {
        "x": {
          "field": "TOTAL KWH",
          "type": "quantitative",
          "scale": { "type": "log" }
        },
        "y": {
          "field": "TOTAL THERMS",
          "type": "quantitative",
          "scale": { "type": "log" }
        },
        "color": { "field": "BUILDING TYPE", "type": "nominal" },
        "tooltip": [
          { "field": "BUILDING TYPE" },
          { "field": "TOTAL KWH" },
          { "field": "TOTAL THERMS" }
        ]
      },
      "title": "Electricity vs Gas Usage by Building Type (Log Scale)"
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_2ba732cf_7fa3_4a59_b41f_faa3d707994f = _curio_output
except NameError:
    result_2ba732cf_7fa3_4a59_b41f_faa3d707994f = None


In [ ]:
__trill_node__ = {
    "id": "e54363aa-e9c8-45d2-a5d0-3938185d4445",
    "type": "DATA_TRANSFORMATION",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_23a15771_4d9a_479f_911f_b6f758850574
    arg = input_0

    df_strip = arg[["BUILDING TYPE", "TOTAL THERMS"]].dropna()

    # Remove large outliers for visualization clarity
    df_strip = df_strip[df_strip["TOTAL THERMS"] < 500_000]

    return df_strip


_curio_output = _curio_node()

try:
    result_e54363aa_e9c8_45d2_a5d0_3938185d4445 = _curio_output
except NameError:
    result_e54363aa_e9c8_45d2_a5d0_3938185d4445 = None


In [ ]:
__trill_node__ = {
    "id": "0503ce30-db2c-4a6f-833f-459969113302",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_e54363aa_e9c8_45d2_a5d0_3938185d4445

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "data": { "name": "df_strip" },
      "mark": "tick",
      "encoding": {
        "x": { "field": "TOTAL THERMS", "type": "quantitative" },
        "y": { "field": "BUILDING TYPE", "type": "nominal" },
        "color": { "field": "BUILDING TYPE", "type": "nominal" },
        "tooltip": [{ "field": "BUILDING TYPE" }, { "field": "TOTAL THERMS" }]
      },
      "title": "Gas Usage Spread by Building Type (Strip Plot)"
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_0503ce30_db2c_4a6f_833f_459969113302 = _curio_output
except NameError:
    result_0503ce30_db2c_4a6f_833f_459969113302 = None


In [ ]:
__trill_node__ = {
    "id": "e62ad390-9e0a-44aa-87f8-644c33974d04",
    "type": "DATA_LOADING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    import pandas as pd
    df = pd.read_csv("/Users/jaideepnutalapati/Documents/GitHub/curio/docs/examples/data/Energy_Usage_5000.csv")
    return df

_curio_output = _curio_node()

try:
    data_e62ad390_9e0a_44aa_87f8_644c33974d04 = _curio_output
except NameError:
    data_e62ad390_9e0a_44aa_87f8_644c33974d04 = None


In [ ]:
__trill_node__ = {
    "id": "42ba84c5-edcf-49b5-ab3b-b5658e018f60",
    "type": "DATA_CLEANING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = data_e62ad390_9e0a_44aa_87f8_644c33974d04
    arg = input_0

    # Filter for KWH month columns
    month_cols = [col for col in arg.columns if col.startswith("KWH ") and "2010" in col]
    required_cols = ["COMMUNITY AREA NAME"] + month_cols

    df = arg[required_cols].dropna()

    # Melt to long format
    df_long = pd.melt(
        df,
        id_vars=["COMMUNITY AREA NAME"],
        value_vars=month_cols,
        var_name="Month",
        value_name="KWH"
    )

    # Extract month name (e.g., "JANUARY")
    df_long["Month"] = df_long["Month"].str.extract(r"KWH (.+?) 2010")[0].str.upper()
    df_long = df_long.dropna(subset=["Month", "KWH", "COMMUNITY AREA NAME"])

    return df_long


_curio_output = _curio_node()

try:
    result_42ba84c5_edcf_49b5_ab3b_b5658e018f60 = _curio_output
except NameError:
    result_42ba84c5_edcf_49b5_ab3b_b5658e018f60 = None


In [ ]:
__trill_node__ = {
    "id": "e7753795-7c01-4773-a9a0-ffeb648cf9bf",
    "type": "DATA_TRANSFORMATION",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_42ba84c5_edcf_49b5_ab3b_b5658e018f60
    arg = input_0

    # Get top 20 communities by average KWH
    top_20 = arg.groupby("COMMUNITY AREA NAME")["KWH"].mean().sort_values(ascending=False).head(20).index

    # Filter the long-form data
    df_top20 = arg[arg["COMMUNITY AREA NAME"].isin(top_20)].copy()
    return df_top20


_curio_output = _curio_node()

try:
    result_e7753795_7c01_4773_a9a0_ffeb648cf9bf = _curio_output
except NameError:
    result_e7753795_7c01_4773_a9a0_ffeb648cf9bf = None


In [ ]:
__trill_node__ = {
    "id": "d537cc76-27b7-41b4-95dc-7ec65ec1ec42",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_e7753795_7c01_4773_a9a0_ffeb648cf9bf

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "params": [
        {
          "name": "commPick",
          "select": {
            "type": "point",
            "fields": ["COMMUNITY AREA NAME"]
          }
        }
      ],
      "vconcat": [
        {
          "title": "Click on a Line to Highlight a Community",
          "width": 650,
          "height": 400,
          "mark": {
            "type": "line",
            "interpolate": "monotone"
          },
          "encoding": {
            "x": {
              "field": "Month",
              "type": "nominal",
              "sort": [
                "JANUARY",
                "FEBRUARY",
                "MARCH",
                "APRIL",
                "MAY",
                "JUNE",
                "JULY",
                "AUGUST",
                "SEPTEMBER",
                "OCTOBER",
                "NOVEMBER",
                "DECEMBER"
              ],
              "axis": { "labelAngle": 45 }
            },
            "y": {
              "field": "KWH",
              "type": "quantitative",
              "title": "Total KWH",
              "scale": { "zero": False }
            },
            "color": {
              "field": "COMMUNITY AREA NAME",
              "type": "nominal",
              "scale": { "scheme": "category20" },
              "legend": { "columns": 2 }
            },
            "opacity": {
              "condition": { "param": "commPick", "value": 1 },
              "value": 0.2
            },
            "tooltip": [
              { "field": "COMMUNITY AREA NAME", "title": "Community" },
              { "field": "Month" },
              { "field": "KWH", "format": ",.0f" }
            ]
          }
        },
        {
          "title": "Average KWH of Selected Community",
          "width": 650,
          "height": 300,
          "mark": "bar",
          "encoding": {
            "y": {
              "field": "COMMUNITY AREA NAME",
              "type": "nominal",
              "sort": "-x"
            },
            "x": {
              "aggregate": "mean",
              "field": "KWH",
              "type": "quantitative",
              "title": "Avg KWH"
            },
            "color": {
              "field": "COMMUNITY AREA NAME",
              "type": "nominal"
            }
          },
          "transform": [{ "filter": { "param": "commPick" } }]
        }
      ]
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_d537cc76_27b7_41b4_95dc_7ec65ec1ec42 = _curio_output
except NameError:
    result_d537cc76_27b7_41b4_95dc_7ec65ec1ec42 = None


In [ ]:
__trill_node__ = {
    "id": "e3b53a07-079f-402c-82c5-69d30fe06b24",
    "type": "DATA_LOADING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    import pandas as pd

    df = pd.read_csv("/Users/jaideepnutalapati/Documents/GitHub/curio/docs/examples/data/Energy_Usage_5000.csv")

    month_cols = [col for col in df.columns if col.startswith("KWH ") and "2010" in col]
    required_cols = ["COMMUNITY AREA NAME"] + month_cols

    df = df[required_cols].dropna()
    return df

_curio_output = _curio_node()

try:
    data_e3b53a07_079f_402c_82c5_69d30fe06b24 = _curio_output
except NameError:
    data_e3b53a07_079f_402c_82c5_69d30fe06b24 = None


In [ ]:
__trill_node__ = {
    "id": "03b3d67d-9bfd-466d-89c3-a616f6951f7d",
    "type": "DATA_CLEANING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = data_e3b53a07_079f_402c_82c5_69d30fe06b24
    arg = input_0

    df_long = pd.melt(
        arg,
        id_vars=["COMMUNITY AREA NAME"],
        value_vars=[col for col in arg.columns if "KWH" in col],
        var_name="Month",
        value_name="KWH"
    )

    df_long["Month"] = df_long["Month"].str.extract(r"KWH (.+?) 2010")[0].str.upper()
    df_long = df_long.dropna(subset=["Month", "KWH", "COMMUNITY AREA NAME"])

    return df_long


_curio_output = _curio_node()

try:
    result_03b3d67d_9bfd_466d_89c3_a616f6951f7d = _curio_output
except NameError:
    result_03b3d67d_9bfd_466d_89c3_a616f6951f7d = None


In [ ]:
__trill_node__ = {
    "id": "2deacd81-afd9-4521-bd35-636b30e7c755",
    "type": "DATA_TRANSFORMATION",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_03b3d67d_9bfd_466d_89c3_a616f6951f7d
    arg = input_0

    top20_names = arg.groupby("COMMUNITY AREA NAME")["KWH"].mean().sort_values(ascending=False).head(20).index
    df_top20 = arg[arg["COMMUNITY AREA NAME"].isin(top20_names)].copy()

    return df_top20


_curio_output = _curio_node()

try:
    result_2deacd81_afd9_4521_bd35_636b30e7c755 = _curio_output
except NameError:
    result_2deacd81_afd9_4521_bd35_636b30e7c755 = None


In [ ]:
__trill_node__ = {
    "id": "d7ba337c-dd7b-49f8-970b-e3114585c58b",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_2deacd81_afd9_4521_bd35_636b30e7c755

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "vconcat": [
        {
          "title": "Monthly Average Energy Usage (Brush to Select Months)",
          "params": [
            {
              "name": "monthBrush",
              "select": {
                "type": "interval",
                "encodings": ["x"]
              }
            }
          ],
          "mark": "bar",
          "encoding": {
            "x": {
              "field": "Month",
              "type": "ordinal",
              "scale": {
                "domain": [
                  "JANUARY",
                  "FEBRUARY",
                  "MARCH",
                  "APRIL",
                  "MAY",
                  "JUNE",
                  "JULY",
                  "AUGUST",
                  "SEPTEMBER",
                  "OCTOBER",
                  "NOVEMBER",
                  "DECEMBER"
                ]
              },
              "axis": {
                "labelAngle": -40,
                "labelFontSize": 11
              }
            },
            "y": {
              "aggregate": "mean",
              "field": "KWH",
              "type": "quantitative",
              "title": "Avg KWH"
            },
            "tooltip": [
              { "field": "Month" },
              { "aggregate": "mean", "field": "KWH", "title": "Avg KWH" }
            ],
            "color": {
              "value": "#4C78A8"
            }
          }
        },
        {
          "title": "Avg KWH by Community (Filtered by Selected Months)",
          "transform": [
            {
              "filter": { "param": "monthBrush" }
            }
          ],
          "mark": "bar",
          "encoding": {
            "y": {
              "field": "COMMUNITY AREA NAME",
              "type": "nominal",
              "sort": "-x"
            },
            "x": {
              "aggregate": "mean",
              "field": "KWH",
              "type": "quantitative",
              "title": "Avg KWH"
            },
            "color": {
              "field": "COMMUNITY AREA NAME",
              "type": "nominal"
            },
            "tooltip": [
              { "field": "COMMUNITY AREA NAME" },
              { "aggregate": "mean", "field": "KWH", "title": "Avg KWH" }
            ]
          }
        }
      ]
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_d7ba337c_dd7b_49f8_970b_e3114585c58b = _curio_output
except NameError:
    result_d7ba337c_dd7b_49f8_970b_e3114585c58b = None


In [ ]:
__trill_node__ = {
    "id": "f5b36cc1-63de-4c10-aca9-c28dd2f3ba3a",
    "type": "DATA_LOADING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    import pandas as pd

    df = pd.read_csv("/Users/jaideepnutalapati/Documents/GitHub/curio/docs/examples/data/Energy_Usage_5000.csv")

    columns_needed = ["AVERAGE STORIES", "AVERAGE BUILDING AGE", "TOTAL KWH"] + [col for col in df.columns if col.startswith("KWH ") and "2010" in col]

    df = df[columns_needed].dropna()
    return df

_curio_output = _curio_node()

try:
    data_f5b36cc1_63de_4c10_aca9_c28dd2f3ba3a = _curio_output
except NameError:
    data_f5b36cc1_63de_4c10_aca9_c28dd2f3ba3a = None


In [ ]:
__trill_node__ = {
    "id": "5c3fa75a-a919-432b-9e95-83e6f1691c8d",
    "type": "DATA_CLEANING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = data_f5b36cc1_63de_4c10_aca9_c28dd2f3ba3a
    arg = input_0

    def story_bracket(stories):
        if stories <= 1:
            return "1 story"
        elif stories == 2:
            return "2 stories"
        elif 3 <= stories <= 5:
            return "3-5 stories"
        elif 6 <= stories <= 10:
            return "6-10 stories"
        else:
            return "11+ stories"

    def age_bracket(age):
        if age <= 20:
            return "0-20 yrs"
        elif age <= 40:
            return "21-40 yrs"
        elif age <= 60:
            return "41-60 yrs"
        elif age <= 80:
            return "61-80 yrs"
        else:
            return "81+ yrs"

    df = arg.copy()

    df["STORY BRACKET"] = df["AVERAGE STORIES"].apply(story_bracket)
    df["AGE BRACKET"] = df["AVERAGE BUILDING AGE"].apply(age_bracket)
    return df


_curio_output = _curio_node()

try:
    result_5c3fa75a_a919_432b_9e95_83e6f1691c8d = _curio_output
except NameError:
    result_5c3fa75a_a919_432b_9e95_83e6f1691c8d = None


In [ ]:
__trill_node__ = {
    "id": "ffa68346-1c37-48f0-9284-7d34a397692f",
    "type": "DATA_TRANSFORMATION",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = result_5c3fa75a_a919_432b_9e95_83e6f1691c8d
    arg = input_0

    import pandas as pd

    df_long = pd.melt(
        arg,
        id_vars=["STORY BRACKET", "AGE BRACKET", "TOTAL KWH"],
        value_vars=[col for col in arg.columns if col.startswith("KWH ")],
        var_name="Month",
        value_name="KWH"
    )

    df_long["Month"] = df_long["Month"].str.extract(r"KWH (.+?) 2010")[0].str.upper()
    df_long = df_long.dropna(subset=["Month", "KWH", "STORY BRACKET", "AGE BRACKET"])

    return df_long


_curio_output = _curio_node()

try:
    result_ffa68346_1c37_48f0_9284_7d34a397692f = _curio_output
except NameError:
    result_ffa68346_1c37_48f0_9284_7d34a397692f = None


In [ ]:
__trill_node__ = {
    "id": "82ef1d71-15c8-481c-a61b-eb172822f7a6",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_ffa68346_1c37_48f0_9284_7d34a397692f

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "params": [
        {
          "name": "storySelect",
          "bind": {
            "input": "select",
            "options": [
              "1 story",
              "2 stories",
              "3-5 stories",
              "6-10 stories",
              "11+ stories"
            ]
          },
          "value": "1 story"
        }
      ],
      "vconcat": [
        {
          "width": 600,
          "title": {
            "text": "Distribution of Total KWH by Age (Box Plot)",
            "align": "center"
          },
          "transform": [{ "filter": "datum['STORY BRACKET'] == storySelect" }],
          "mark": "boxplot",
          "encoding": {
            "x": {
              "field": "AGE BRACKET",
              "type": "nominal",
              "sort": ["0-20 yrs", "21-40 yrs", "41-60 yrs", "61-80 yrs", "81+ yrs"]
            },
            "y": {
              "field": "TOTAL KWH",
              "type": "quantitative",
              "title": "Total KWH"
            },
            "color": {
              "field": "AGE BRACKET",
              "type": "nominal",
              "legend": {
                "orient": "right",
                "anchor": "middle",
                "direction": "vertical"
              }
            },
            "tooltip": [{ "field": "AGE BRACKET" }, { "field": "TOTAL KWH" }]
          }
        },
        {
          "width": 600,
          "title": {
            "text": "Monthly Avg KWH Trend by Age (for Selected Stories)",
            "align": "center"
          },
          "transform": [{ "filter": "datum['STORY BRACKET'] == storySelect" }],
          "mark": { "type": "line", "point": True },
          "encoding": {
            "x": {
              "field": "Month",
              "type": "ordinal",
              "sort": [
                "JANUARY",
                "FEBRUARY",
                "MARCH",
                "APRIL",
                "MAY",
                "JUNE",
                "JULY",
                "AUGUST",
                "SEPTEMBER",
                "OCTOBER",
                "NOVEMBER",
                "DECEMBER"
              ]
            },
            "y": {
              "aggregate": "mean",
              "field": "KWH",
              "type": "quantitative",
              "title": "Avg Monthly KWH"
            },
            "color": {
              "field": "AGE BRACKET",
              "type": "nominal",
              "legend": {
                "orient": "right",
                "anchor": "middle",
                "direction": "vertical"
              }
            },
            "tooltip": [
              { "field": "Month" },
              { "aggregate": "mean", "field": "KWH" },
              { "field": "AGE BRACKET" }
            ]
          }
        }
      ],
      "config": {
        "concat": { "align": "center" }
      }
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_82ef1d71_15c8_481c_a61b_eb172822f7a6 = _curio_output
except NameError:
    result_82ef1d71_15c8_481c_a61b_eb172822f7a6 = None
